# 07 — Exception Handling Best Practices

This notebook is the **quality and design layer** of Exception Handling.

You have already learned:
1. Errors & Exceptions
2. `try` and `except`
3. Multiple Exceptions
4. `else` and `finally`
5. Raising Exceptions
6. Custom Exceptions

This notebook does **not introduce new exception-handling syntax**. Instead, it focuses on writing exception-handling code that is **clean, safe, meaningful, predictable, and maintainable**.

## 1. Introduction

Exception handling is not only about preventing a program from crashing.

There is a major difference between:

- code that merely prevents a crash
- code that handles an error correctly and professionally

Good exception handling should be:

- **Specific** — catch what you actually know how to handle
- **Meaningful** — provide useful messages and actions
- **Minimal** — keep exception-handling areas focused
- **Predictable** — preserve normal error behavior when appropriate
- **Maintainable** — make the code easy to understand and debug

### Poor approach

In [ ]:
try:
    number = int(input("Enter a number: "))
except:
    print("Something went wrong")

### Better approach

In [ ]:
try:
    number = int(input("Enter a number: "))
except ValueError:
    print("Please enter a valid integer.")

The second version communicates exactly which error is expected and gives the user a useful message.

## 2. Catch Specific Exceptions

One of the most important rules is:

> **Catch the exception you actually know how to handle.**

Avoid using a broad exception when a specific exception describes the problem.

In [ ]:
# Too broad
try:
    number = int(input("Enter a number: "))
except Exception:
    print("Invalid input")

In [ ]:
# Specific
try:
    number = int(input("Enter a number: "))
except ValueError:
    print("Please enter a valid number.")

In [ ]:
try:
    result = 10 / number
except ZeroDivisionError:
    print("Cannot divide by zero.")

Specific handlers are easier to understand, less likely to hide unrelated programming errors, and make future maintenance safer.

## 3. Avoid Bare `except`

There are three different levels worth distinguishing:

```python
except:
```

```python
except Exception:
```

```python
except ValueError:
```

A bare `except` catches almost everything, including exceptions such as `KeyboardInterrupt` and `SystemExit`. This can make debugging and program control difficult.

In [ ]:
# Bad
try:
    x = int(input("Enter number: "))
except:
    print("Error")

In [ ]:
# Better
try:
    x = int(input("Enter number: "))
except ValueError:
    print("Invalid number.")

Use a specific exception whenever possible. A bare `except` should generally be avoided.

## 4. Avoid Overusing `except Exception`

`Exception` can be appropriate at an **application boundary**, such as a top-level program, logging boundary, API boundary, or background job.

Inside ordinary business logic, prefer specific exceptions.

In [ ]:
# Potentially appropriate at an application boundary
try:
    process_data()
except Exception as e:
    log_error(e)

Examples of places where broad handling can be justified:

- application-level error handling
- logging systems
- API boundaries
- background jobs
- top-level programs

The important question is not simply *"Can I catch Exception?"* but *"Why am I catching it here, and what can I meaningfully do with it?"*

## 5. Don't Silently Ignore Exceptions

This pattern is dangerous:

```python
try:
    data = load_data()
except Exception:
    pass
```

The error disappears, the program may continue with invalid state, and debugging becomes difficult.

In [ ]:
# Better
try:
    data = load_data()
except FileNotFoundError:
    print("Data file was not found.")

Sometimes ignoring a **specific** exception is intentional when the desired final state has already been achieved.

In [ ]:
import os

try:
    os.remove("temporary.txt")
except FileNotFoundError:
    pass

Here, ignoring `FileNotFoundError` is intentional because the desired final state is simply:

> The file does not exist.

## 6. Keep `try` Blocks Small

A `try` block should contain only the operations that can reasonably produce the exception you want to handle.

Large `try` blocks make it unclear which operation failed.

In [ ]:
# Poor design
try:
    name = input("Enter name: ")
    age = int(input("Enter age: "))
    salary = float(input("Enter salary: "))
    result = salary / age
    print(name)
    save_data(name, age, salary)
except ValueError:
    print("Invalid value")

In [ ]:
# Better
name = input("Enter name: ")

try:
    age = int(input("Enter age: "))
except ValueError:
    print("Age must be an integer.")
    age = None

try:
    salary = float(input("Enter salary: "))
except ValueError:
    print("Salary must be a number.")
    salary = None

> **The smaller the `try` block, the easier it is to understand exactly what failed.**

## 7. Don't Use Exceptions for Normal Control Flow

Exceptions are designed for **exceptional situations**, not ordinary program logic.

If a normal condition can be expressed clearly with `if`, `in`, `.get()`, or validation, prefer those tools.

In [ ]:
# Poor when the index check is normal logic
try:
    value = numbers[10]
except IndexError:
    value = None

In [ ]:
# Clearer normal control flow
if len(numbers) > 10:
    value = numbers[10]
else:
    value = None

In [ ]:
# Poor
try:
    value = dictionary["name"]
except KeyError:
    value = "Unknown"

In [ ]:
# Natural dictionary operation
value = dictionary.get("name", "Unknown")

Exceptions should not unnecessarily replace:

- `if`
- `in`
- `.get()`
- validation
- ordinary conditions

## 8. Write Meaningful Error Messages

A generic message such as `"Error"` provides little information.

A useful error message should explain:

1. What went wrong
2. What value caused it, when appropriate
3. What was expected

In [ ]:
raise ValueError("Error")

In [ ]:
raise ValueError("Age cannot be negative.")

In [ ]:
age = -5

raise ValueError(
    f"Age must be between 0 and 120. Received: {age}"
)

## 9. Don't Expose Sensitive Information

Detailed exceptions can sometimes contain information that should not be shown to users.

Avoid exposing information such as:

- passwords
- API keys
- database credentials
- sensitive file paths
- personal information
- internal system details

In [ ]:
# Avoid displaying potentially sensitive exception details
try:
    operation()
except Exception as e:
    print(e)

In [ ]:
# Safer user-facing message
try:
    operation()
except Exception:
    print("An unexpected error occurred.")

Detailed diagnostic information can instead be logged securely inside the application. User-facing messages should reveal only what is appropriate.

## 10. `print()` vs Logging

For simple learning programs, `print()` is often enough.

Professional applications commonly use Python's `logging` module because logging provides structured levels and can be configured for different environments.

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Division by zero occurred.")

In [ ]:
import logging

logging.basicConfig(level=logging.ERROR)

try:
    result = 10 / 0
except ZeroDivisionError:
    logging.error("Division by zero occurred.")

Basic logging levels:

- `DEBUG`
- `INFO`
- `WARNING`
- `ERROR`
- `CRITICAL`

You do not need a full logging lesson here. Remember:

> `print()` is useful for simple programs, while logging is more appropriate for production applications.

## 11. Preserve Exception Information

When catching an exception, do not unnecessarily destroy the original information.

In [ ]:
# Less useful
try:
    number = int("abc")
except ValueError:
    raise ValueError("Something went wrong")

In [ ]:
# Preserve the original exception information
try:
    number = int("abc")
except ValueError as e:
    print("Invalid number:", e)

When translating an exception into a domain-specific exception, use exception chaining so the original cause remains available.

In [ ]:
class DataProcessingError(Exception):
    pass


try:
    number = int("abc")
except ValueError as e:
    raise DataProcessingError("Could not process input data") from e

## 12. Re-Raising Exceptions Correctly

A bare `raise` inside an `except` block re-raises the exception currently being handled.

In [ ]:
def process_data():
    raise ValueError("Invalid data")


try:
    process_data()
except ValueError:
    print("Logging error...")
    raise

Re-raising is useful when you want to:

1. log the error
2. perform another action
3. let another layer handle the exception

## 13. Exception Chaining

Use:

```python
raise NewException(...) from e
```

when you want to translate an original exception while preserving its cause.

In [ ]:
class DataProcessingError(Exception):
    pass


try:
    number = int("abc")
except ValueError as e:
    raise DataProcessingError("Invalid data received") from e

Compare:

```python
raise DataProcessingError("Invalid data")
```

with:

```python
raise DataProcessingError("Invalid data") from e
```

The second version explicitly preserves the original exception as the cause.

## 14. Use `finally` for Cleanup

Cleanup code should run whether an operation succeeds or fails.

Typical cleanup includes:

- closing files
- releasing resources
- closing connections
- cleaning temporary resources

In [ ]:
file = None

try:
    file = open("data.txt", "r")
    data = file.read()
except FileNotFoundError:
    print("File not found.")
finally:
    if file:
        file.close()

For files, a `with` statement is usually preferable because it manages cleanup automatically.

In [ ]:
with open("data.txt", "r") as file:
    data = file.read()

This notebook only uses `with` as a practical cleanup pattern; context managers are a separate topic.

## 15. Validate Input Early

A useful professional principle is:

> **Fail early — validate early.**

Reject invalid data before performing calculations or changing program state.

In [ ]:
# Poor
def calculate_discount(price, discount):
    result = price - (price * discount / 100)

    if price < 0:
        raise ValueError("Price cannot be negative.")

    return result

In [ ]:
# Better
def calculate_discount(price, discount):
    if price < 0:
        raise ValueError("Price cannot be negative.")

    if not 0 <= discount <= 100:
        raise ValueError("Discount must be between 0 and 100.")

    return price - (price * discount / 100)

Benefits:

- invalid state is rejected early
- debugging is easier
- functions are clearer
- invalid calculations are prevented

## 16. Handle Exceptions at the Appropriate Layer

Real applications often contain layers:

```text
User Input
    ↓
Function
    ↓
Business Logic
    ↓
Database
```

Different layers may have different responsibilities.

In [ ]:
def read_age():
    try:
        return int(input("Enter age: "))
    except ValueError:
        raise ValueError("Age must be an integer.")

In [ ]:
try:
    age = read_age()
except ValueError as e:
    print(e)

The function understands how the input conversion failed. The caller understands how the error should be presented to the user.

> **The layer that knows how to handle an error should generally handle it.**

## 17. Don't Catch an Exception You Cannot Handle

If the current function does not know how to recover, translate, log, or meaningfully handle an exception, it may be better to let it propagate.

In [ ]:
# If this layer has no meaningful recovery:
def process_data():
    process_database()

Avoid handlers that merely catch an exception and replace it with a vague message without adding useful behavior.

## 18. Custom Exceptions vs Built-in Exceptions

Use a built-in exception when it naturally describes the problem.

In [ ]:
raise ValueError("Age cannot be negative.")

Use a custom exception when the application domain benefits from a meaningful category.

In [ ]:
class InsufficientBalanceError(Exception):
    pass


balance = 500

if 1000 > balance:
    raise InsufficientBalanceError(
        "Insufficient account balance."
    )

Do not create a new exception class for every small problem.

Custom exceptions should provide meaningful domain value.

## 19. Don't Over-Catch Exceptions

A broad handler can hide unrelated bugs.

In [ ]:
# Too broad
try:
    data = process_data()
except Exception:
    print("Processing failed")

In [ ]:
def process_data():
    result = 10 / 0
    return result

The broad handler above could hide the `ZeroDivisionError`.

Prefer handling only the expected problem at that point.

In [ ]:
try:
    data = load_data()
except FileNotFoundError:
    print("Input file not found.")

## 20. Avoid Deeply Nested Exception Handling

Deeply nested `try` blocks are difficult to read and maintain.

In [ ]:
# Difficult to maintain
try:
    try:
        try:
            value = int(input("Enter number: "))
        except ValueError:
            print("Invalid number")
    except TypeError:
        print("Type error")
except Exception:
    print("Unknown error")

In [ ]:
# Simpler structure
try:
    value = int(input("Enter number: "))
except ValueError:
    print("Invalid number")
except TypeError:
    print("Type error")

Unnecessary nesting should be simplified whenever possible.

## 21. Use `else` for the Success Path

`else` can clearly separate successful execution from the operation that may fail.

In [ ]:
try:
    number = int(input("Enter number: "))
except ValueError:
    print("Invalid number.")
else:
    print("Successfully converted:", number)

This separates:

- code that may fail → `try`
- error handling → `except`
- successful execution → `else`

That separation often makes the code easier to understand.

## 22. Don't Return from `finally`

Avoid changing a function's result from `finally`.

In [ ]:
def example():
    try:
        return 10
    finally:
        return 20


print(example())

The result is `20` because the `finally` return overrides the earlier return.

In [ ]:
def example():
    try:
        10 / 0
    finally:
        return "Success"


print(example())

A return in `finally` can even suppress an exception.

> Use `finally` for cleanup, not for changing the function's result.

## 23. Don't Hide Bugs with Exception Handling

Exception handling should improve reliability, not hide programming mistakes.

In [ ]:
# Dangerous
try:
    result = calculate_result()
except Exception:
    result = 0

In [ ]:
def calculate_result():
    return unknown_variable  # NameError

The broad handler can make a program appear to work while silently producing incorrect data.

A handler should not be used as a substitute for fixing programming errors.

## 24. Use Exceptions to Maintain Valid Program State

Exceptions can prevent objects from entering invalid states.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive.")

        if amount > self.balance:
            raise ValueError("Insufficient balance.")

        self.balance -= amount

This connects exception handling with:

- OOP
- properties
- validation
- custom exceptions

The important idea is that invalid state should be rejected rather than silently accepted.

## 25. Before vs After — Practical Comparison

### Poor Code

The following version works for some cases, but it has several design problems.

In [ ]:
def divide():
    try:
        a = int(input("Enter a: "))
        b = int(input("Enter b: "))

        result = a / b

        print(result)

    except:
        print("Error")

Problems:

- bare `except`
- no specific error handling
- input and calculation are mixed together
- unclear error message
- difficult to extend

### Better Code

In [ ]:
def divide():
    try:
        a = int(input("Enter a: "))
        b = int(input("Enter b: "))
        result = a / b

    except ValueError:
        print("Please enter valid integers.")

    except ZeroDivisionError:
        print("The second number cannot be zero.")

    else:
        print("Result:", result)

The improved version identifies the expected exceptions and provides a clear success path.

## 26. Practical Example — File Processing

In [ ]:
def read_file(filename):
    try:
        with open(filename, "r") as file:
            return file.read()

    except FileNotFoundError:
        print(f"File '{filename}' was not found.")

    except PermissionError:
        print("You do not have permission to read this file.")

This example demonstrates:

- specific exceptions
- a focused `try` block
- `with` for resource management
- meaningful messages
- handling only expected problems

## 27. Practical Example — Data Validation

In [ ]:
def register_student(name, age, marks):
    if not name:
        raise ValueError("Name cannot be empty.")

    if age < 0:
        raise ValueError("Age cannot be negative.")

    if not 0 <= marks <= 100:
        raise ValueError("Marks must be between 0 and 100.")

    return {
        "name": name,
        "age": age,
        "marks": marks
    }

In [ ]:
try:
    student = register_student("Komal", 22, 85)
    print(student)

except ValueError as e:
    print("Validation error:", e)

This demonstrates:

- validation
- `raise`
- meaningful errors
- caller-side handling

## 28. Practical Example — Layered Exception Handling

A simple application architecture can be viewed as:

```text
Input Layer
    ↓
Validation Layer
    ↓
Business Logic
    ↓
Exception Handling
```

In [ ]:
def calculate_average(marks):
    if not marks:
        raise ValueError("Marks list cannot be empty.")

    return sum(marks) / len(marks)

In [ ]:
try:
    average = calculate_average([])
except ValueError as e:
    print("Input error:", e)

The function is responsible for identifying invalid input. The caller decides how the error should be presented.

This separation keeps reusable functions independent of a particular user interface.

## 29. Common Exception-Handling Anti-Patterns

### Anti-pattern 1 — Bare `except`
```python
except:
```

### Anti-pattern 2 — Catch everything without a reason
```python
except Exception:
```

### Anti-pattern 3 — Silent failure
```python
except:
    pass
```

### Anti-pattern 4 — Huge `try` block
```python
try:
    # 30 lines of unrelated code
```

### Anti-pattern 5 — Generic error messages
```python
print("Error")
```

### Anti-pattern 6 — Hiding bugs
```python
except Exception:
    return None
```

### Anti-pattern 7 — Using exceptions for normal conditions

Using exceptions instead of simple `if` checks or other natural operations.

### Anti-pattern 8 — Returning from `finally`
```python
finally:
    return value
```

### Anti-pattern 9 — Exposing sensitive information

Never expose passwords, API keys, credentials, or inappropriate internal details.

### Anti-pattern 10 — Unnecessary custom exceptions

Not every validation error requires a new exception class.

## 30. Exception Handling Checklist

Before finalizing exception handling, ask:

```text
☐ Am I catching a specific exception?
☐ Do I actually know how to handle this exception?
☐ Is my try block small?
☐ Is the error message meaningful?
☐ Am I hiding an actual programming bug?
☐ Am I silently ignoring an exception?
☐ Should I use logging instead of print?
☐ Do I need to re-raise the exception?
☐ Should I preserve the original cause?
☐ Do I need finally for cleanup?
☐ Am I using exceptions for an exceptional situation?
☐ Should I use a built-in or custom exception?
☐ Am I exposing sensitive information?
```

## 31. Best Practices Summary

| Practice | Recommendation |
|---|---|
| Catching | Catch specific exceptions |
| `except` | Avoid bare `except` |
| `Exception` | Use only when justified |
| `try` block | Keep it small |
| Messages | Make them meaningful |
| `pass` | Don't silently ignore errors |
| Logging | Prefer logging for applications |
| Validation | Validate early |
| `raise` | Use when invalid state occurs |
| Re-raise | Use `raise` when appropriate |
| Chaining | Use `raise ... from e` |
| Cleanup | Use `finally` / context managers |
| Custom exceptions | Use when domain-specific |
| Control flow | Don't use exceptions for ordinary conditions |
| Security | Don't expose sensitive information |
| Debugging | Don't hide programming bugs |

## 32. Final Practical Project — Robust Bank Account System

This mini project combines the major exception-handling practices from the section.

### Requirements

```text
BankAccount
│
├── Deposit
├── Withdraw
├── Balance checking
├── Input validation
├── Custom exceptions
├── Specific exception handling
├── Meaningful messages
└── Proper exception propagation
```

In [ ]:
class InsufficientBalanceError(Exception):
    pass


class InvalidAmountError(Exception):
    pass


class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise InvalidAmountError(
                "Deposit amount must be greater than zero."
            )

        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise InvalidAmountError(
                "Withdrawal amount must be greater than zero."
            )

        if amount > self.balance:
            raise InsufficientBalanceError(
                "Insufficient account balance."
            )

        self.balance -= amount

    def get_balance(self):
        return self.balance

In [ ]:
account = BankAccount(5000)

try:
    account.deposit(1000)
    account.withdraw(2000)

except InvalidAmountError as e:
    print("Invalid amount:", e)

except InsufficientBalanceError as e:
    print("Transaction failed:", e)

else:
    print("Transaction successful.")

finally:
    print("Current balance:", account.get_balance())

This project connects the progression:

```text
OOP
 ↓
Custom Exceptions
 ↓
raise
 ↓
try / except
 ↓
else / finally
 ↓
Best Practices
```

# 🎯 What You Should Understand After This Notebook

By the end of `07_Exception_Handling_Best_Practices.ipynb`, you should be able to look at exception-handling code and ask:

> **"Is this code merely preventing an error, or is it handling the error professionally?"**

The progression through this section is:

```text
Errors & Exceptions
        ↓
try / except
        ↓
Multiple Exceptions
        ↓
else / finally
        ↓
raise
        ↓
Custom Exceptions
        ↓
Best Practices
```

This notebook acts as the **quality/design layer** over the first six Exception Handling notebooks.

### Next file

The final file in this section is:

**`08_Practical_Exception_Handling.ipynb`**

That notebook can be more project-oriented, combining exception handling into realistic Python applications rather than introducing more theory.